# 02 Paper Fig. 3 Style Scattering Rates

This notebook maps the paper Fig. 3 calculation onto DeePTB APIs: conduction-band inverse lifetimes versus electronic energy, using a q mesh around Gamma, `T = 300 K`, `mu = 100 meV`, and Gaussian smearing. The default q/k sampling is reduced for runtime; the parameter cell shows the paper-scale values.


In [ ]:
from pathlib import Path
import json
import numpy as np

from dptb.nn.dftbsk import DFTBSK
from dptb.postprocess.unified import TBSystem
from dptb.postprocess.unified.eph import Phonons

ROOT = Path.cwd()
DATA = ROOT / "data"
WORK = ROOT / "work"
WORK.mkdir(exist_ok=True)

SK_DATA = DATA / "skf" / "matsci-0-3"
STRUCTURE = ROOT / "graphene.vasp"
PHONOPY_YAML = DATA / "graphene" / "phonons" / "phonopy_disp.yaml"
FORCE_SETS = DATA / "graphene" / "phonons" / "FORCE_SETS"

for path, message in [
    (SK_DATA / "C-C.skf", "The bundled matsci-0-3 C-C.skf file is required."),
    (STRUCTURE, "The bundled graphene.vasp structure is required."),
    (PHONOPY_YAML, "The bundled phonopy_disp.yaml file is required."),
    (FORCE_SETS, "The bundled FORCE_SETS file is required."),
]:
    if not path.exists():
        raise FileNotFoundError(f"{path} is missing. {message}")

BASIS = {"C": ["2s", "2p"]}
model = DFTBSK(
    basis=BASIS,
    skdata=str(SK_DATA),
    overlap=True,
    dtype="float64",
    device="cpu",
    interp_method="smooth_intp",
    r_max={"C": 6.349479778742587},
)
model.eval()
system = TBSystem(data=str(STRUCTURE), calculator=model)


def regularize_tiny_negative_frequencies(phonons, tol=1e-3):
    frequencies = np.array(phonons.frequencies, copy=True)
    min_frequency = float(np.min(frequencies))
    if min_frequency < -tol:
        raise ValueError(
            f"Found phonon frequency {min_frequency} THz below tolerance {-tol} THz; "
            "this looks like a real imaginary mode, not acoustic numerical noise."
        )
    clipped = int(np.count_nonzero(frequencies < 0.0))
    frequencies[frequencies < 0.0] = 0.0
    if clipped:
        phonons = Phonons(
            qpoints=phonons.qpoints,
            frequencies=frequencies,
            eigenvectors=phonons.eigenvectors,
            masses=phonons.masses,
            cell=phonons.cell,
            scaled_positions=phonons.scaled_positions,
            metadata={
                **phonons.metadata,
                "negative_frequency_regularization": "clipped_to_zero",
                "negative_frequency_tolerance_thz": tol,
                "negative_frequency_min_before_clip_thz": min_frequency,
                "negative_frequency_clipped_count": clipped,
            },
        )
    return phonons


def get_eigenvalues_array(kpoints):
    _, evals = system.get_eigenvalues(k_points=np.asarray(kpoints, dtype=float))
    if hasattr(evals, "detach"):
        evals = evals.detach().cpu().numpy()
    return np.asarray(evals, dtype=float)


def json_default(value):
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return float(value)
    if isinstance(value, (np.bool_,)):
        return bool(value)
    return str(value)

print(system.atoms)
print(system.model.name, system.model.basis)


In [ ]:
K_POINT = np.array([-2.0 / 3.0, -1.0 / 3.0, 0.0])
G_POINT = np.array([0.0, 0.0, 0.0])
CONDUCTION_BAND = 2
VALENCE_BAND = 1


def q_grid_around_gamma(n=9, qmax=0.0665):
    axis = np.linspace(-qmax, qmax, n)
    qx, qy = np.meshgrid(axis, axis, indexing="xy")
    qpoints = np.column_stack([qx.ravel(), qy.ravel(), np.zeros(qx.size)])
    return qpoints, qx, qy


def choose_k_near_k_for_energy(target_ev=0.30, nscan=41):
    alphas = np.linspace(0.0, 0.18, nscan)
    kpoints = K_POINT[None, :] + alphas[:, None] * (G_POINT - K_POINT)[None, :]
    evals = get_eigenvalues_array(kpoints)
    dirac = float(evals[0, CONDUCTION_BAND])
    energies = evals[:, CONDUCTION_BAND] - dirac
    idx = int(np.argmin(np.abs(energies - target_ev)))
    return kpoints[idx], dirac, energies[idx], evals[idx]


def acoustic_in_plane_modes(phonons, q_index):
    eig = np.asarray(phonons.eigenvectors[q_index])
    freqs = np.asarray(phonons.frequencies[q_index])
    acoustic = np.argsort(freqs)[:3]
    z_weight = np.sum(np.abs(eig[acoustic, :, 2]) ** 2, axis=1)
    in_plane = acoustic[np.argsort(z_weight)[:2]]
    ordered = in_plane[np.argsort(freqs[in_plane])]
    return {"TA": int(ordered[0]), "LA": int(ordered[1])}


def cell_area_ang2(atoms):
    cell = np.asarray(atoms.cell.array, dtype=float)
    return float(np.linalg.norm(np.cross(cell[0], cell[1])))


In [ ]:
from dptb.postprocess.unified.eph import compute_linewidth_mesh
from dptb.postprocess.unified.eph.data import EPCMeshData
from dptb.utils.constants import HBAR_EV_S

Q_GRID_N = 7
K_SAMPLE_N = 9
Q_MAX_FRACTIONAL = 0.0665
PAPER_Q_GRID_N = 200
TEMPERATURE_EV = 8.617333262145e-5 * 300.0
CHEMICAL_POTENTIAL_EV = 0.100
SIGMA_EV = 0.003
DISPLACEMENT = 1e-3
BANDS = [1, 2]
CONDUCTION_BAND_POSITION = BANDS.index(CONDUCTION_BAND)

qpoints, qx, qy = q_grid_around_gamma(Q_GRID_N, Q_MAX_FRACTIONAL)
q_weights = np.full(qpoints.shape[0], 1.0 / qpoints.shape[0])
alphas = np.linspace(0.01, 0.16, K_SAMPLE_N)
kpoints = K_POINT[None, :] + alphas[:, None] * (G_POINT - K_POINT)[None, :]
all_evals = get_eigenvalues_array(kpoints)
dirac_energy = float(get_eigenvalues_array(K_POINT[None, :])[0, CONDUCTION_BAND])
energy_axis = all_evals[:, CONDUCTION_BAND] - dirac_energy

phonons = regularize_tiny_negative_frequencies(
    Phonons.from_phonopy_file(PHONOPY_YAML, qpoints=qpoints, force_sets_filename=str(FORCE_SETS))
)

print("k samples", kpoints.shape)
print("q samples", qpoints.shape, "paper target", f"{PAPER_Q_GRID_N}x{PAPER_Q_GRID_N}")
print("energy range above Dirac [eV]", float(energy_axis.min()), float(energy_axis.max()))


## Compute mode-resolved linewidths

The linewidth postprocess implements the SERTA energy-conservation structure used by paper Eq. 15. We convert linewidth to inverse lifetime through DeePTB's convention `tau = hbar / (2 linewidth)`.


In [ ]:
epc = system.eph.compute_coupling(
    kpoints=kpoints,
    phonons=phonons,
    bands=BANDS,
    displacement=DISPLACEMENT,
    output_npz=WORK / "fig3_scattering_epc_data.npz",
)
mesh = EPCMeshData.from_epc_data(
    epc,
    kpoint_weights=np.full(kpoints.shape[0], 1.0 / kpoints.shape[0]),
    qpoint_weights=q_weights,
    metadata={"mesh_kind": "fig3_reduced_intravalley_q_mesh"},
)
linewidth = compute_linewidth_mesh(
    mesh,
    chemical_potential=dirac_energy + CHEMICAL_POTENTIAL_EV,
    temperature=TEMPERATURE_EV,
    sigma=SIGMA_EV,
    broadening="gaussian",
    mode_resolved=True,
)
linewidth.save_npz(WORK / "fig3_mode_resolved_linewidth.npz")

nearest_nonzero_q = int(np.argmin(np.where(np.linalg.norm(qpoints[:, :2], axis=1) > 1e-12, np.linalg.norm(qpoints[:, :2], axis=1), np.inf)))
mode_map = acoustic_in_plane_modes(phonons, nearest_nonzero_q)
mode_rates_ps = {}
for label, mode in mode_map.items():
    gamma_ev = linewidth.linewidth[:, CONDUCTION_BAND_POSITION, mode]
    mode_rates_ps[label] = 2.0 * gamma_ev / HBAR_EV_S / 1e12

total_rate_ps = 2.0 * linewidth.linewidth[:, CONDUCTION_BAND_POSITION, :].sum(axis=-1) / HBAR_EV_S / 1e12
print("mode map", mode_map)
print("rates ps^-1", {k: v.tolist() for k, v in mode_rates_ps.items()})


## Figure: inverse lifetime versus energy

This is the paper Fig. 3 style output. For a converged comparison, increase `Q_GRID_N` toward 200 and densify `K_SAMPLE_N`; this notebook keeps a reduced grid so it can be run as an example.


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(5.4, 3.4), dpi=140)
for label, values in mode_rates_ps.items():
    ax.plot(energy_axis, values, marker="o", label=label)
ax.plot(energy_axis, total_rate_ps, marker="s", color="black", linestyle="--", label="TA+LA+all modes")
ax.set_xlabel("conduction-band energy above Dirac [eV]")
ax.set_ylabel("inverse lifetime [ps^-1]")
ax.set_title("Graphene intravalley scattering, T=300 K, mu=100 meV")
ax.grid(alpha=0.25)
ax.legend()
fig.tight_layout()
fig.savefig(WORK / "figure_02_fig3_scattering_rates.png", dpi=200)
plt.show()


In [ ]:
summary = {
    "figure": "paper_fig3_style_scattering_rates",
    "q_grid_n": Q_GRID_N,
    "paper_q_grid_n": PAPER_Q_GRID_N,
    "k_sample_n": K_SAMPLE_N,
    "temperature_ev": TEMPERATURE_EV,
    "temperature_k": 300.0,
    "chemical_potential_above_dirac_ev": CHEMICAL_POTENTIAL_EV,
    "sigma_ev": SIGMA_EV,
    "mode_map": mode_map,
    "energy_axis_ev": energy_axis,
    "rate_ps_inverse": mode_rates_ps,
    "total_rate_ps_inverse": total_rate_ps,
    "note": "Reduced-grid DeePTB-native reproduction of paper Fig. 3 conventions.",
}
(WORK / "fig3_scattering_rates_summary.json").write_text(json.dumps(summary, indent=2, default=json_default))
print(json.dumps({k: v for k, v in summary.items() if k not in {"rate_ps_inverse", "total_rate_ps_inverse"}}, indent=2, default=json_default))
